# Task 1 — Best Buy Canada Review Scraping and Sentiment Analysis

This notebook uses Playwright to inspect publicly visible product reviews and BeautifulSoup for HTML parsing. Before running, set `PRODUCT_URL` to a Best Buy Canada product with at least 50 reviews and inspect the current page structure. Selectors change; the notebook deliberately avoids fabricated review records.

**Compliance:** respect Best Buy's current terms and robots guidance, use low request rates, do not access private or authenticated data, do not evade CAPTCHA or access controls, and stop if the site blocks automation.

In [10]:
import re, time, hashlib
from datetime import datetime
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

PRODUCT_URL = "PASTE_PUBLIC_BESTBUY_CANADA_PRODUCT_URL_HERE"
SOURCE = "Best Buy Canada"
OUT_CSV = Path("task1_reviews_with_sentiment.csv")

SELECTORS = {
    "review_card": "[data-testid*='review'], .review-item, .review",
    "title": "[data-testid*='review-title'], .review-title, h3",
    "body": "[data-testid*='review-text'], .review-text, .review-content",
    "date": "time, [data-testid*='date']",
    "rating": "[aria-label*='out of 5'], [aria-label*='star'], .rating",
    "reviewer": "[data-testid*='reviewer'], .reviewer-name, .author"
}


## 1. Collect public review cards with pagination

In [11]:
def first_text(card, selector):
    el = card.select_one(selector)
    if not el:
        return ""
    return el.get("datetime", "") if el.name == "time" and el.get("datetime") else el.get_text(" ", strip=True)

def parse_rating(card):
    el = card.select_one(SELECTORS["rating"])
    if not el:
        return None
    raw = " ".join([el.get("aria-label", ""), el.get_text(" ", strip=True)])
    match = re.search(r"([1-5](?:\\.\\d+)?)", raw)
    try:
        value = float(match.group(1)) if match else None
        return value if value is not None and 0 <= value <= 5 else None
    except (ValueError, AttributeError):
        return None

def parse_date(raw):
    if not raw:
        return ""
    raw = raw.strip()
    for fmt in ("%Y-%m-%d", "%Y-%m-%dT%H:%M:%S.%fZ", "%m/%d/%Y", "%B %d, %Y", "%b %d, %Y"):
        try:
            return datetime.strptime(raw[:26] if "T" in raw else raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    try:
        return pd.to_datetime(raw, errors="raise").strftime("%Y-%m-%d")
    except Exception:
        return ""

def scrape_reviews(product_url, max_pages=30, delay_seconds=2):
    if "bestbuy.ca" not in product_url or "PASTE_PUBLIC" in product_url:
        raise ValueError("Set PRODUCT_URL to a real public Best Buy Canada product URL.")
    rows, seen = [], set()
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(product_url, wait_until="domcontentloaded", timeout=60000)
        page.wait_for_timeout(2500)
        for page_no in range(max_pages):
            html = page.content()
            low = html.lower()
            if "captcha" in low or "verify you are human" in low or "access denied" in low:
                print("Anti-bot challenge/block detected. Stopping; do not attempt to bypass it.")
                break
            soup = BeautifulSoup(html, "html.parser")
            cards = soup.select(SELECTORS["review_card"])
            for card in cards:
                title = first_text(card, SELECTORS["title"])
                body = first_text(card, SELECTORS["body"])
                date = parse_date(first_text(card, SELECTORS["date"]))
                rating = parse_rating(card)
                reviewer = first_text(card, SELECTORS["reviewer"])
                if not (title or body):
                    continue
                identity = "|".join([title, body, date, str(rating), reviewer])
                key = hashlib.sha256(identity.encode("utf-8")).hexdigest()[:16]
                if key in seen:
                    continue
                seen.add(key)
                rows.append({
                    "Primary Key": key, "Review Title": title, "Review Body Text": body,
                    "Date": date, "Star Rating": rating, "Source Platform": SOURCE,
                    "Reviewer Name": reviewer, "Product URL": product_url
                })
            # Inspect actual page to replace this with the current visible "Show More"/next control.
            more = page.get_by_role("button", name=re.compile(r"show more|load more|next", re.I))
            if more.count() == 0 or not more.first.is_visible() or not more.first.is_enabled():
                break
            try:
                more.first.click(timeout=5000)
                page.wait_for_timeout(delay_seconds * 1000)
            except PlaywrightTimeoutError:
                break
        browser.close()
    return pd.DataFrame(rows)

# Run only after setting PRODUCT_URL and confirming selectors against the current public page.
# reviews = scrape_reviews(PRODUCT_URL)
# print("Reviews collected:", len(reviews))
# reviews.to_csv("task1_reviews_raw.csv", index=False)


## 2. Sentiment scoring and aspect tags

In [ ]:
# VADER is a lexicon baseline;
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

ASPECT_TERMS = {
    "Good Design & Quality": ["design", "quality", "build", "colour", "color", "durable"],
    "Performance & Functionality": ["works", "performance", "feature", "function", "fast", "effective"],
    "Ease of Use & Setup": ["easy", "setup", "install", "assemble", "simple"],
    "Value & Price": ["price", "value", "cost", "expensive", "cheap"],
    "Delivery & Packaging": ["delivery", "shipping", "package", "damaged"],
    "Customer Service": ["service", "support", "return", "warranty"]
}
def sentiment_label(compound):
    if compound >= 0.05: return "Positive"
    if compound <= -0.05: return "Negative"
    return "Neutral"

def categorize_review(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    sentiment = sentiment_label(score)
    lower = str(text).lower()
    aspects = [a for a, terms in ASPECT_TERMS.items() if any(term in lower for term in terms)]
    if not aspects:
        aspects = ["General Satisfaction" if sentiment == "Positive" else "General Concern" if sentiment == "Negative" else "General Feedback"]
    return score, sentiment, [f"{a} ({'Pos' if sentiment == 'Positive' else 'Neg' if sentiment == 'Negative' else 'Neu'})" for a in aspects]

# Example after scraping:
# reviews["Combined Text"] = reviews["Review Title"].fillna("") + " " + reviews["Review Body Text"].fillna("")
# scored = reviews["Combined Text"].apply(categorize_review)
# reviews["Sentiment Score"] = scored.apply(lambda x: x[0])
# reviews["Sentiment"] = scored.apply(lambda x: x[1])
# reviews["Sentiment Tags"] = scored.apply(lambda x: str(x[2]))
# reviews.to_csv("task1_reviews_with_sentiment.csv", index=False)


## 3. Basic findings for the report